# Heretic on Qwen3-4B in Colab

This notebook clones the Rowan Dauria Heretic fork, installs it into the Colab runtime, writes a small `config.toml`, and runs Heretic on Qwen3-4B with Qwen thinking mode disabled. Disabling thinking avoids generating `<think>...</think>` blocks during Heretic's repeated generations, which should save wall time.

Use a GPU runtime: **Runtime -> Change runtime type -> GPU**. For long runs, mount Google Drive so Optuna checkpoints survive runtime resets.

In [ ]:
# @title Runtime knobs
REPO_URL = "https://github.com/rowan-dauria/heretic.git"  # @param {type:"string"}
REPO_BRANCH = "codex/heretic-touched-layers"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-4B"  # @param {type:"string"}

# Use bnb_4bit for Colab T4/L4 safety. On A100/H100, "none" is faster if VRAM is ample.
QUANTIZATION = "none"  # @param ["bnb_4bit", "none"]

# Start smaller for a smoke run, then increase once the setup is proven.
N_TRIALS = 40  # @param {type:"integer"}
N_STARTUP_TRIALS = 12  # @param {type:"integer"}
MAX_RESPONSE_LENGTH = 80  # @param {type:"integer"}
MAX_BATCH_SIZE = 32  # @param {type:"integer"}
BATCH_SIZE = 0  # @param {type:"integer"}

# Smaller prompt slices make quick test runs much cheaper. Increase for final runs.
TRAIN_SLICE = "train[:200]"  # @param {type:"string"}
EVAL_SLICE = "test[:80]"  # @param {type:"string"}

USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_RUN_DIR = "/content/drive/MyDrive/heretic-qwen3-4b"  # @param {type:"string"}
LOCAL_RUN_DIR = "/content/heretic-qwen3-4b"  # @param {type:"string"}

ENABLE_THINKING = False

## Check the GPU

If this does not show a CUDA GPU, switch the runtime before continuing.

In [2]:
!nvidia-smi

Wed Jun 17 19:35:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Clone and install the fork

The notebook installs the local checkout in editable mode so it uses your fork rather than the PyPI release. On the first pass this cell also pins a compatible NumPy/SciPy/scikit-learn stack and restarts the runtime once; after the restart, rerun the notebook from the top and the marker file will skip the reinstall.

In [3]:
from pathlib import Path
import os
import shutil

repo_dir = Path("/content/heretic")
setup_marker = Path("/content/.heretic_colab_setup_complete")

# Avoid deleting the current working directory when rerunning after a restart.
os.chdir("/content")

if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone --branch "$REPO_BRANCH" "$REPO_URL" "$repo_dir" || (git clone "$REPO_URL" "$repo_dir" && cd "$repo_dir" && git checkout "$REPO_BRANCH")
%cd /content/heretic

if not setup_marker.exists():
    !python -m pip install -q --upgrade pip
    !python -m pip install -q --force-reinstall --no-cache-dir "numpy==2.2.6" "scipy==1.15.3" "scikit-learn==1.7.2"
    !python -m pip install -q -e .
    setup_marker.write_text("done\n")
    print("Installed Heretic and repaired the numeric stack. Restarting the runtime now; rerun the notebook from the top after Colab reconnects.")
    os.kill(os.getpid(), 9)
else:
    !python -m pip install -q --no-deps -e .
    print("Setup marker found; using existing repaired package stack.")


Cloning into '/content/heretic'...
remote: Enumerating objects: 1010, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 1010 (delta 50), reused 39 (delta 26), pack-reused 939 (from 3)
Receiving objects: 100% (1010/1010), 1.41 MiB | 3.52 MiB/s, done.
Resolving deltas: 100% (595/595), done.
/content/heretic
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for heretic-llm (pyproject.toml) ... done
Setup marker found; using existing repaired package stack.


## Optional Hugging Face login

`Qwen/Qwen3-4B` and the default datasets are public, so this is usually unnecessary. Log in if you switch to a gated/private model, want to upload results, or hit Hub rate limits.

In [4]:
try:
    from google.colab import userdata
    from huggingface_hub import login

    token = userdata.get("HF_TOKEN")
    if token:
        login(token=token)
        print("Logged in with HF_TOKEN from Colab secrets.")
    else:
        print("No HF_TOKEN Colab secret found; continuing anonymously.")
except Exception as exc:
    print(f"Skipping Hugging Face login: {exc}")

Skipping Hugging Face login: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.


## Prepare the run directory and config

The important line is `enable_thinking = false`. Heretic's patched fork passes this into `tokenizer.apply_chat_template(...)`, so Qwen3 should not spend tokens on hidden reasoning blocks.

In [15]:
from pathlib import Path
import textwrap

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    run_dir = Path(DRIVE_RUN_DIR)
else:
    run_dir = Path(LOCAL_RUN_DIR)

run_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir = run_dir / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

config = f"""
model = "{MODEL_ID}"
dtypes = ["bfloat16", "float16", "auto"]
quantization = "{QUANTIZATION}"
device_map = "auto"
offload_outputs_to_cpu = true

batch_size = {BATCH_SIZE}
max_batch_size = {MAX_BATCH_SIZE}
max_response_length = {MAX_RESPONSE_LENGTH}

enable_thinking = {str(ENABLE_THINKING).lower()}
response_prefix = ""

n_trials = {N_TRIALS}
n_startup_trials = {N_STARTUP_TRIALS}
study_checkpoint_dir = "{checkpoint_dir}"

# Save LoRA adapters by default if you choose to export after optimization.
export_strategy = "adapter"

[good_prompts]
dataset = "mlabonne/harmless_alpaca"
split = "{TRAIN_SLICE}"
column = "text"

[bad_prompts]
dataset = "mlabonne/harmful_behaviors"
split = "{TRAIN_SLICE}"
column = "text"

[good_evaluation_prompts]
dataset = "mlabonne/harmless_alpaca"
split = "{EVAL_SLICE}"
column = "text"

[bad_evaluation_prompts]
dataset = "mlabonne/harmful_behaviors"
split = "{EVAL_SLICE}"
column = "text"
"""

(run_dir / "config.toml").write_text(textwrap.dedent(config).strip() + "\n")
print(f"Run directory: {run_dir}")
print((run_dir / "config.toml").read_text())

Run directory: /content/heretic-qwen3-4b
model = "Qwen/Qwen3-4B"
dtypes = ["bfloat16", "float16", "auto"]
quantization = "bnb_4bit"
device_map = "auto"
offload_outputs_to_cpu = true

batch_size = 0
max_batch_size = 32
max_response_length = 80

enable_thinking = false
response_prefix = ""

n_trials = 40
n_startup_trials = 12
study_checkpoint_dir = "/content/heretic-qwen3-4b/checkpoints"

# Save LoRA adapters by default if you choose to export after optimization.
export_strategy = "adapter"

[good_prompts]
dataset = "mlabonne/harmless_alpaca"
split = "train[:200]"
column = "text"

[bad_prompts]
dataset = "mlabonne/harmful_behaviors"
split = "train[:200]"
column = "text"

[good_evaluation_prompts]
dataset = "mlabonne/harmless_alpaca"
split = "test[:80]"
column = "text"

[bad_evaluation_prompts]
dataset = "mlabonne/harmful_behaviors"
split = "test[:80]"
column = "text"



## Sanity-check the chat template

This prints the rendered prompt tail. With thinking disabled, Qwen should not include an opening `<think>` instruction in the generated assistant prefix. If this cell still reports a NumPy import error, restart the runtime once manually and rerun from the top.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
rendered = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Give me a one-sentence greeting."}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)
print(rendered[-600:])

<|im_start|>user
Give me a one-sentence greeting.<|im_end|>
<|im_start|>assistant
<think>

</think>




## Run Heretic

This cell is interactive after optimization: select a trial, then choose whether to save an adapter, chat, benchmark, or exit. If the run is interrupted, rerun the notebook with the same `run_dir`; Heretic will offer to resume from the checkpoint.

In [ ]:
import os
os.chdir(run_dir)
!pwd
!PYTORCH_ALLOC_CONF=expandable_segments:True heretic

/content/heretic-qwen3-4b
█░█░█▀▀░█▀▄░█▀▀░▀█▀░█░█▀▀  v1.4.0
█▀█░█▀▀░█▀▄░█▀▀░░█░░█░█░░  https://heretic-project.org
▀░▀░▀▀▀░▀░▀░▀▀▀░░▀░░▀░▀▀▀  https://github.com/p-e-w/heretic

Detected 1 CUDA device(s) (22.03 GB total VRAM)
CUDA Version: 12.8
Driver Version: 580.82.07
* CUDA 0: NVIDIA L4 (22.03 GB)

Loading model Qwen/Qwen3-4B...
* Trying dtype bfloat16...
Loading weights:   0% 1/398 [00:00<01:17,  5.11it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 398/398 [00:02<00:00, 168.74it/s]
* Quantized to 4-bit precision
* LoRA adapters initialized (target types: down_proj, o_proj)
* Transformer model with 36 layers
* Abliterable components:
  * attn.o_proj: 36 modules total
  * mlp.down_proj: 36 modules total

Resident system RAM: 1.77 GB
Allocated GPU VRAM: 2.49 GB

## Export notes

With `export_strategy = "adapter"`, choose **Save the model to a local folder** after selecting a trial and provide a path inside `run_dir`, for example `/content/heretic-qwen3-4b/qwen3-4b-heretic-adapter`. Adapter export is much lighter than merging, especially if the run used 4-bit quantization.